# Evaluation

Require both `generated_answer.csv` and `generated_answer.json` at the cwd.

In [ ]:
!pip install -q ragas langchain-openai langchain-huggingface
!pip list | grep -E 'ragas|langchain'

langchain                                1.2.14
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.23
langchain-huggingface                    1.2.1
langchain-openai                         1.1.12
langchain-text-splitters                 1.1.1
ragas                                    0.4.3


In [ ]:
import pandas as pd
import ast
import itertools
import json
import os
from google.colab import userdata

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextRelevance, AspectCritic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_3269/127400755.py:9: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, ContextRelevance, AspectCritic
/tmp/ipykernel_3269/127400755.py:9: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.co

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = userdata.get('OPENAI_BASE_URL')

In [ ]:
configs = {
    "llms": ["mistral", "llama"],
    "chunk_strategies": [
        "SentenceSplitter(chunk_size=128, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=256, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=512, chunk_overlap=10)",
        "SemanticSplitterNodeParser",
        "SentenceWindowNodeParser"
    ]
}

In [ ]:
with open("generated_answer.json") as f:
    test_answers = json.load(f)

In [ ]:
judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
))
embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

/tmp/ipykernel_3269/3294206712.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOpenAI(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3269/3294206712.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(


In [ ]:
task_completion = AspectCritic(
    name="task_completion",
    definition=(
        "Return 1 if the response fully resolves the user's query, "
        "addresses all aspects of the question, and provides a complete answer "
        "that matches the intent of the reference answer. "
        "Return 0 if the response is incomplete, off-topic, or fails to "
        "adequately address the user's question."
    ),
)

In [ ]:
total_configs = list(itertools.product(*configs.values()))
eval_results = {}

for i, (conf_llm, conf_chunk_strat) in enumerate(total_configs):
    print(f'evaluating answers on config {i}/{len(total_configs)}')
    samples = []
    for row in test_answers:
        answer = row[f"{conf_llm}:{conf_chunk_strat}:answer"]
        contexts = row[f"{conf_llm}:{conf_chunk_strat}:contexts"]
        samples.append(SingleTurnSample(
            user_input=row["question"],
            response=answer,
            retrieved_contexts=contexts,
            reference=row["reference_answer"],
        ))

    eval_dataset = EvaluationDataset(samples=samples)

    eval_ = evaluate(
        dataset=eval_dataset,
        metrics=[Faithfulness(), ResponseRelevancy(), ContextRelevance(), task_completion],
        llm=judge_llm,
        embeddings=embeddings,
    )

    eval_results[f"{conf_llm}:{conf_chunk_strat}"] = eval_.to_pandas()

print('evaluation complete')

evaluating answers on config 0/10


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

evaluating answers on config 1/10


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

evaluating answers on config 2/10


...

evaluating answers on config 8/10


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

evaluating answers on config 9/10


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

evaluation complete


In [ ]:
df_test_ans = pd.read_csv('./generated_answer.csv')
eval_cols = ['faithfulness', 'answer_relevancy', 'nv_context_relevance', 'task_completion']

for i, (conf_llm, conf_chunk_strat) in enumerate(total_configs):
    conf_key = f'{conf_llm}:{conf_chunk_strat}'
    eval_result = eval_results[conf_key][eval_cols]
    eval_result.columns = [f'{conf_key}:{col}' for col in eval_result]
    df_test_ans = pd.concat([df_test_ans, eval_result], axis=1)

df_test_ans.to_csv('./evaluation.csv', index=False)